# 04 — Análise de Clientes
**Olist Analytics Hub | Etapa 4 de 7**

**Objetivo:** Entender quem são os clientes da Olist — onde estão, quando compram, com que frequência e qual o valor que geram ao longo do tempo.

---
### Perguntas que este notebook responde
1. Qual a distribuição geográfica dos clientes por estado?
2. Qual o padrão de compra por hora e dia da semana?
3. Qual a taxa de recompra na plataforma?
4. Qual o LTV básico (valor médio por cliente único)?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

from src.data_cleaner import load_processed
from src.visualizations import setup_style, save_fig, format_brl, bar_chart
from src.config import COLORS

setup_style()
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✓ Ambiente configurado.')

In [ ]:
df = load_processed('orders_enriched')
delivered = df[df['is_delivered']].copy()

print(f'Pedidos entregues: {len(delivered):,}')
print(f'Clientes únicos: {delivered["customer_unique_id"].nunique():,}')
print(f'Estados representados: {delivered["customer_state"].nunique()}')

## 1. Distribuição Geográfica por Estado

In [ ]:
state_stats = (
    delivered
    .groupby('customer_state')
    .agg(
        pedidos=('order_id', 'count'),
        clientes_unicos=('customer_unique_id', 'nunique'),
        receita=('revenue', 'sum'),
        ticket_medio=('revenue', 'mean'),
        nota_media=('review_score', 'mean'),
    )
    .sort_values('pedidos', ascending=False)
    .reset_index()
)
state_stats['share_pct'] = state_stats['pedidos'] / state_stats['pedidos'].sum() * 100

print('Top 10 estados por volume de pedidos:')
display(
    state_stats.head(10)
    .style
    .format({'receita': 'R$ {:,.0f}', 'ticket_medio': 'R$ {:,.2f}',
             'nota_media': '{:.2f}', 'share_pct': '{:.1f}%'})
    .background_gradient(subset=['pedidos'], cmap='Blues')
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10_states = state_stats.head(10)

# Pedidos por estado
axes[0].barh(top10_states['customer_state'], top10_states['pedidos'],
             color=COLORS['primary'], alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('Top 10 Estados — Volume de Pedidos')
axes[0].set_xlabel('Pedidos')
for i, (_, row) in enumerate(top10_states.iterrows()):
    axes[0].text(row['pedidos'] * 1.01, i, f'{row["share_pct"]:.1f}%', va='center', fontsize=8)

# Ticket médio por estado
top10_ticket = state_stats.nlargest(10, 'ticket_medio')
axes[1].barh(top10_ticket['customer_state'], top10_ticket['ticket_medio'],
             color=COLORS['accent'], alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('Top 10 Estados — Ticket Médio')
axes[1].set_xlabel('Ticket Médio (R$)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

plt.tight_layout()
save_fig('04_estados')
plt.show()

In [ ]:
# Mapa choropleth interativo (Plotly)
# Mapeamento de siglas para nomes completos (geojson padrão do Brasil)
state_name_map = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas',
    'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Distrito Federal', 'ES': 'Espírito Santo',
    'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso', 'MS': 'Mato Grosso do Sul',
    'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}

state_map_df = state_stats.copy()
state_map_df['estado_nome'] = state_map_df['customer_state'].map(state_name_map)

fig_map = px.choropleth(
    state_map_df,
    locations='customer_state',
    color='pedidos',
    hover_name='estado_nome',
    hover_data={'pedidos': True, 'receita': ':,.0f', 'ticket_medio': ':,.2f', 'nota_media': ':.2f'},
    title='Distribuição de Pedidos por Estado',
    color_continuous_scale='Blues',
    locationmode='geojson-id',
    scope='south america',
)
fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(paper_bgcolor='white', height=500)
fig_map.show()

## 2. Padrão de Compra por Hora e Dia da Semana

In [ ]:
dias_ordem = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_pt = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
dia_map = dict(zip(dias_ordem, dias_pt))

delivered['dia_semana_pt'] = delivered['purchase_day_of_week'].map(dia_map)

heatmap_data = (
    delivered
    .groupby(['purchase_day_of_week', 'purchase_hour'])['order_id']
    .count()
    .unstack(fill_value=0)
    .reindex(dias_ordem)
)
heatmap_data.index = dias_pt

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.3,
    annot=False,
    cbar_kws={'shrink': 0.8, 'label': 'Pedidos'},
)
ax.set_title('Concentração de Pedidos: Hora × Dia da Semana', fontsize=14)
ax.set_xlabel('Hora do Dia')
ax.set_ylabel('')
plt.tight_layout()
save_fig('04_heatmap_hora_dia')
plt.show()

pico = heatmap_data.stack().idxmax()
print(f'Combinação de maior volume: {pico[0]}, {pico[1]}h')

## 3. Taxa de Recompra

In [ ]:
# Contagem de pedidos por cliente único
pedidos_por_cliente = (
    delivered
    .groupby('customer_unique_id')['order_id']
    .count()
    .rename('n_pedidos')
)

total_clientes = len(pedidos_por_cliente)
recompradores = (pedidos_por_cliente > 1).sum()
taxa_recompra = recompradores / total_clientes

dist_pedidos = pedidos_por_cliente.value_counts().sort_index().head(10)

print(f'Total de clientes únicos: {total_clientes:,}')
print(f'Clientes com mais de 1 pedido: {recompradores:,} ({taxa_recompra:.1%})')
print(f'Máx. pedidos por cliente: {pedidos_por_cliente.max()}')
print()
print('Distribuição de pedidos por cliente:')
for n, count in dist_pedidos.items():
    print(f'  {n} pedido(s): {count:,} clientes ({count/total_clientes:.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza: 1 vs 2+ pedidos
labels = ['1 pedido\n(clientes únicos)', '2+ pedidos\n(recompradores)']
sizes = [total_clientes - recompradores, recompradores]
colors_pie = [COLORS['primary'], COLORS['secondary']]
wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, autopct='%1.1f%%',
    colors=colors_pie, startangle=140, pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
axes[0].set_title('Taxa de Recompra de Clientes')

# Distribuição de pedidos por cliente
axes[1].bar(
    dist_pedidos.index.astype(str),
    dist_pedidos.values,
    color=COLORS['primary'], alpha=0.85
)
axes[1].set_title('Distribuição: Pedidos por Cliente')
axes[1].set_xlabel('Número de Pedidos')
axes[1].set_ylabel('Clientes')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

plt.tight_layout()
save_fig('04_recompra')
plt.show()

## 4. LTV Básico (Lifetime Value)

In [ ]:
ltv = (
    delivered
    .groupby('customer_unique_id')
    .agg(
        n_pedidos=('order_id', 'count'),
        ltv=('revenue', 'sum'),
        primeiro_pedido=('order_purchase_timestamp', 'min'),
        ultimo_pedido=('order_purchase_timestamp', 'max'),
    )
    .reset_index()
)
ltv['ciclo_vida_dias'] = (ltv['ultimo_pedido'] - ltv['primeiro_pedido']).dt.days

print('Estatísticas de LTV por cliente único:')
print(f'  LTV médio: {format_brl(ltv["ltv"].mean())}')
print(f'  LTV mediana: {format_brl(ltv["ltv"].median())}')
print(f'  LTV total: {format_brl(ltv["ltv"].sum())}')
print(f'  LTV p90: {format_brl(ltv["ltv"].quantile(0.90))}')
print(f'  LTV p99: {format_brl(ltv["ltv"].quantile(0.99))}')

# Concentração de receita (Pareto)
ltv_sorted = ltv.sort_values('ltv', ascending=False).reset_index(drop=True)
ltv_sorted['ltv_cumulativo_pct'] = ltv_sorted['ltv'].cumsum() / ltv_sorted['ltv'].sum() * 100
ltv_sorted['clientes_pct'] = (ltv_sorted.index + 1) / len(ltv_sorted) * 100

pareto_20 = ltv_sorted.loc[ltv_sorted['clientes_pct'] <= 20, 'ltv_cumulativo_pct'].max()
print(f'\n  Os top 20% dos clientes representam {pareto_20:.1f}% da receita total (Pareto)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma de LTV
p95_ltv = ltv['ltv'].quantile(0.95)
axes[0].hist(ltv.loc[ltv['ltv'] <= p95_ltv, 'ltv'], bins=60,
             color=COLORS['primary'], alpha=0.85, edgecolor='white')
axes[0].axvline(ltv['ltv'].mean(), color=COLORS['secondary'], linestyle='--', linewidth=2,
                label=f'Média: {format_brl(ltv["ltv"].mean())}')
axes[0].axvline(ltv['ltv'].median(), color=COLORS['accent'], linestyle='--', linewidth=2,
                label=f'Mediana: {format_brl(ltv["ltv"].median())}')
axes[0].set_title('Distribuição de LTV por Cliente (até p95)')
axes[0].set_xlabel('LTV (R$)')
axes[0].set_ylabel('Clientes')
axes[0].legend()

# Curva de Pareto
axes[1].plot(ltv_sorted['clientes_pct'], ltv_sorted['ltv_cumulativo_pct'],
             color=COLORS['primary'], linewidth=2.5)
axes[1].axvline(20, color='gray', linestyle=':', linewidth=1.5)
axes[1].axhline(pareto_20, color='gray', linestyle=':', linewidth=1.5)
axes[1].fill_between(ltv_sorted['clientes_pct'], ltv_sorted['ltv_cumulativo_pct'],
                     alpha=0.1, color=COLORS['primary'])
axes[1].set_title('Curva de Pareto — Concentração de Receita')
axes[1].set_xlabel('% de Clientes (ordenado por LTV desc.)')
axes[1].set_ylabel('% da Receita Acumulada')
axes[1].text(22, pareto_20 - 5, f'{pareto_20:.1f}%', fontsize=10, color='gray')

# LTV médio por estado (top 10)
ltv_estado = (
    delivered
    .groupby(['customer_unique_id', 'customer_state'])['revenue']
    .sum()
    .reset_index()
    .groupby('customer_state')['revenue']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
axes[2].barh(ltv_estado.index, ltv_estado.values, color=COLORS['secondary'], alpha=0.85)
axes[2].invert_yaxis()
axes[2].set_title('LTV Médio por Estado — Top 10')
axes[2].set_xlabel('LTV Médio (R$)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

plt.tight_layout()
save_fig('04_ltv')
plt.show()

## 5. Cohort Mensal de Novos Clientes

In [ ]:
# Primeiro pedido de cada cliente
primeiro_pedido = (
    delivered
    .groupby('customer_unique_id')['order_purchase_timestamp']
    .min()
    .reset_index()
    .rename(columns={'order_purchase_timestamp': 'primeiro_pedido'})
)
primeiro_pedido['cohort_mes'] = primeiro_pedido['primeiro_pedido'].dt.to_period('M')

novos_por_mes = (
    primeiro_pedido
    .groupby('cohort_mes')['customer_unique_id']
    .count()
    .reset_index()
)
novos_por_mes['mes'] = novos_por_mes['cohort_mes'].dt.to_timestamp()
novos_por_mes = novos_por_mes.iloc[1:-1]  # Remove meses parciais

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(novos_por_mes['mes'], novos_por_mes['customer_unique_id'],
                alpha=0.2, color=COLORS['secondary'])
ax.plot(novos_por_mes['mes'], novos_por_mes['customer_unique_id'],
        color=COLORS['secondary'], linewidth=2.5, marker='o', markersize=5)
ax.set_title('Novos Clientes por Mês (primeira compra)')
ax.set_xlabel('')
ax.set_ylabel('Novos Clientes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.xticks(rotation=45)
plt.tight_layout()
save_fig('04_novos_clientes')
plt.show()

## 6. Sumário e Insights

| Métrica | Valor |
|---------|-------|
| Clientes únicos | *(calculado acima)* |
| Taxa de recompra | *(calculado acima — tipicamente ~3%)* |
| Estado dominante | SP (~42% dos pedidos) |
| LTV médio | *(calculado acima)* |
| Pareto | Top 20% clientes → ~X% da receita |
| Pico de compras | Dias úteis, 10h–16h |

### Observações importantes
- A baixa taxa de recompra é característica do dataset (a maioria dos clientes compra uma única vez no período analisado)
- A concentração em SP reflete o peso demográfico e de infraestrutura logística do estado
- A curva de Pareto mostra a concentração de valor em poucos clientes — público-alvo prioritário para campanhas de retenção

### Próximo Notebook
`05_segmentacao_rfm.ipynb` — segmentação formal dos clientes com K-Means (Recência, Frequência, Valor).